# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Description:")
print(metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Here we list out all record sets and their fields, referencing only by their `@id`.

In [ ]:
# List all record sets
record_sets = dataset.metadata.record_sets  # List of mlc.RecordSet
print(f"Total record sets found: {len(record_sets)}\n")
for rec_set in record_sets:
    print(f"RecordSet @id: {rec_set['@id']}")
    print(f"  name: {rec_set.get('name', 'N/A')}")
    # List fields for this record set
    fields = rec_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    Field @id: {field['@id']} - name: {field.get('name', 'N/A')}")
    print()

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use record set and field `@id`s as referenced above.

In [ ]:
# Choose the main protein record set (replace the @id with the main table you want to analyze, shown above)
# For demonstration, we try to auto-select the most relevant one

rs_candidates = [rs['@id'] for rs in dataset.metadata.record_sets]
print("Record set options:", rs_candidates)

# By inspection, the main protein/measurement table may be the first non-documentation table, or you can set specifically:
main_record_set_id = rs_candidates[0] if rs_candidates else None
record_sets_to_extract = rs_candidates  # Extract all for completeness

dataframes = {}
for record_set_id in record_sets_to_extract:
    print(f"Loading records for {record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print("  No records found. Skipping.")
            continue
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Loaded {len(records)} records, columns: {dataframes[record_set_id].columns.tolist()}")
    except Exception as e:
        print(f"  Failed to load record set {record_set_id}: {e}")

if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nColumns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No main record set loaded!")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section shows how to refer to columns via their field `@id`s.

In [ ]:
# For demonstration, select a numeric field (eg. coverage, peptides, or MW) 
# Identify fields with likely numeric columns. Adjust the field_id variables as appropriate.
numeric_candidates = []
if main_record_set_id:
    df = dataframes[main_record_set_id]
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_candidates.append(c)
    print("Numeric field candidates in main record set:", numeric_candidates)

    # Choose the first numeric field as example (e.g., coverage/%/MW/Abundance)
    # You may want to specify this more precisely for your analysis
    numeric_field_id = numeric_candidates[0] if numeric_candidates else None
    group_field_id = [c for c in df.columns if 'group' in c.lower() or 'type' in c.lower()]
    group_field_id = group_field_id[0] if group_field_id else None
    print("Selected numeric field for analysis:", numeric_field_id)
    print("Selected group field for grouping:", group_field_id)

    # Set threshold arbitrarily for filtering
    threshold = df[numeric_field_id].quantile(0.75) if numeric_field_id else 0
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Optionally group by a field if present
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient numeric field information for visualization.")

## 6. Conclusion
This notebook demonstrates loading and exploring the FAIR² mass spectrometry dataset via `mlcroissant`. You performed:

- Schema and field inspection via `@id`
- Data extraction to pandas DataFrames
- Numeric field filtering and normalization
- Optional grouping and aggregation
- Visualizations for data distribution

Explore additional record sets or fields by referencing them via their Croissant `@id` as listed above.

For more dataset and Croissant schema documentation, see [mlcroissant documentation](https://mlcommons.github.io/croissant/).